In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 2263d8de-392d-4c8f-889f-d87e20dc2bad, 3, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.functions import (
    col, lit, when, coalesce, countDistinct, sum as spark_sum,
    min as spark_min, max as spark_max, avg, round,
    greatest, unix_timestamp, current_timestamp
)

def overwrite_table(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    print(f"Created {table_name}: {df.count()} rows")

# A shipment can have multiple products.
# Use the strictest shared temperature range for its risk assessment.
shipment_limits = (
    spark.table("silver_shipment_items")
    .join(
        spark.table("silver_products").select(
            "product_id", "min_temp_c", "max_temp_c"
        ),
        "product_id",
        "left"
    )
    .groupBy("shipment_id")
    .agg(
        spark_max("min_temp_c").alias("allowed_min_temp_c"),
        spark_min("max_temp_c").alias("allowed_max_temp_c"),
        spark_sum(
            col("quantity") * col("declared_unit_value_inr")
        ).alias("shipment_value_inr"),
        countDistinct("product_id").alias("product_count")
    )
    .withColumn(
        "temperature_range_conflict",
        col("allowed_min_temp_c") > col("allowed_max_temp_c")
    )
)

# Compare every sensor reading with the permitted shipment temperature range.
sensor_risk_summary = (
    spark.table("silver_sensor_readings")
    .join(shipment_limits, "shipment_id", "left")
    .withColumn(
        "temperature_breach_flag",
        when(
            (col("temperature_c") < col("allowed_min_temp_c")) |
            (col("temperature_c") > col("allowed_max_temp_c")),
            1
        ).otherwise(0)
    )
    .withColumn(
        "temperature_deviation_c",
        greatest(
            lit(0.0),
            col("allowed_min_temp_c") - col("temperature_c"),
            col("temperature_c") - col("allowed_max_temp_c")
        )
    )
    .groupBy("shipment_id")
    .agg(
        spark_sum("temperature_breach_flag").alias("temperature_breach_count"),
        spark_max("temperature_deviation_c").alias("max_temperature_deviation_c"),
        spark_min("temperature_c").alias("minimum_observed_temp_c"),
        spark_max("temperature_c").alias("maximum_observed_temp_c"),
        spark_min("battery_pct").alias("minimum_battery_pct"),
        spark_min("captured_at_utc").alias("first_sensor_reading_utc"),
        spark_max("captured_at_utc").alias("latest_sensor_reading_utc")
    )
)

shipment_risk = (
    spark.table("silver_shipments")
    .join(
        spark.table("silver_carriers").select(
            "carrier_id",
            "carrier_name",
            "baseline_on_time_pct",
            "baseline_temperature_compliance_pct"
        ),
        "carrier_id",
        "left"
    )
    .join(shipment_limits, "shipment_id", "left")
    .join(sensor_risk_summary, "shipment_id", "left")
    .withColumn(
        "late_arrival_hours",
        when(
            col("actual_arrival_utc").isNotNull(),
            greatest(
                lit(0.0),
                (
                    unix_timestamp("actual_arrival_utc") -
                    unix_timestamp("expected_arrival_utc")
                ) / lit(3600.0)
            )
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "temperature_risk_points",
        when(coalesce(col("temperature_breach_count"), lit(0)) >= 3, 55)
        .when(coalesce(col("temperature_breach_count"), lit(0)) >= 1, 30)
        .otherwise(0)
    )
    .withColumn(
        "delivery_risk_points",
        when(col("late_arrival_hours") > 6, 20)
        .when(col("late_arrival_hours") > 0, 10)
        .otherwise(0)
    )
    .withColumn(
        "priority_risk_points",
        when(col("priority") == "Critical", 10).otherwise(0)
    )
    .withColumn(
        "carrier_risk_points",
        when(col("baseline_temperature_compliance_pct") < 93, 10)
        .when(col("baseline_temperature_compliance_pct") < 96, 5)
        .otherwise(0)
    )
    .withColumn(
        "battery_risk_points",
        when(col("minimum_battery_pct") < 60, 5).otherwise(0)
    )
    .withColumn(
        "range_conflict_risk_points",
        when(col("temperature_range_conflict"), 15).otherwise(0)
    )
    .withColumn(
        "risk_score",
        col("temperature_risk_points") +
        col("delivery_risk_points") +
        col("priority_risk_points") +
        col("carrier_risk_points") +
        col("battery_risk_points") +
        col("range_conflict_risk_points")
    )
    .withColumn(
        "risk_level",
        when(col("risk_score") >= 65, "Critical")
        .when(col("risk_score") >= 40, "High")
        .when(col("risk_score") >= 20, "Medium")
        .otherwise("Low")
    )
    .withColumn("risk_calculated_at_utc", current_timestamp())
)

gold_shipment_risk = shipment_risk.select(
    "shipment_id",
    "shipment_status",
    "priority",
    "carrier_id",
    "carrier_name",
    "origin_warehouse_id",
    "destination_warehouse_id",
    "planned_departure_utc",
    "expected_arrival_utc",
    "actual_arrival_utc",
    "allowed_min_temp_c",
    "allowed_max_temp_c",
    "shipment_value_inr",
    "product_count",
    "temperature_breach_count",
    "max_temperature_deviation_c",
    "minimum_observed_temp_c",
    "maximum_observed_temp_c",
    "minimum_battery_pct",
    "latest_sensor_reading_utc",
    "late_arrival_hours",
    "risk_score",
    "risk_level",
    "risk_calculated_at_utc"
)

overwrite_table(gold_shipment_risk, "gold_shipment_risk")

StatementMeta(, 7635f63e-1822-4c09-beea-52bc59275e2b, 5, Finished, Available, Finished, False)

Created gold_shipment_risk: 36 rows


In [4]:
display(
    spark.sql("""
        SELECT
            risk_level,
            COUNT(*) AS shipment_count,
            ROUND(AVG(risk_score), 1) AS average_risk_score
        FROM gold_shipment_risk
        GROUP BY risk_level
        ORDER BY
            CASE risk_level
                WHEN 'Critical' THEN 1
                WHEN 'High' THEN 2
                WHEN 'Medium' THEN 3
                ELSE 4
            END
    """)
)

display(
    spark.sql("""
        SELECT
            shipment_id,
            carrier_name,
            temperature_breach_count,
            maximum_observed_temp_c,
            allowed_max_temp_c,
            late_arrival_hours,
            risk_score,
            risk_level
        FROM gold_shipment_risk
        ORDER BY risk_score DESC, shipment_id
    """)
)

StatementMeta(, 7635f63e-1822-4c09-beea-52bc59275e2b, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 943ecf2f-55e6-4c45-a461-3145e78c6603)

SynapseWidget(Synapse.DataFrame, bbd5def4-acb1-40ac-a108-ba00b6460fff)

In [5]:
from pyspark.sql.functions import (
    col,
    lit,
    when,
    coalesce,
    countDistinct,
    sum as spark_sum,
    avg,
    round,
    current_timestamp
)

# ============================================================
# GOLD: CARRIER PERFORMANCE
# ============================================================

gold_carrier_performance = (
    spark.table("gold_shipment_risk")
    .groupBy(
        "carrier_id",
        "carrier_name"
    )
    .agg(
        countDistinct("shipment_id").alias("shipment_count"),

        spark_sum(
            when(
                col("shipment_status") == "Delivered",
                1
            ).otherwise(0)
        ).alias("delivered_shipment_count"),

        spark_sum(
            when(
                col("risk_level").isin("High", "Critical"),
                1
            ).otherwise(0)
        ).alias("high_risk_shipment_count"),

        spark_sum(
            coalesce(col("temperature_breach_count"), lit(0))
        ).alias("total_temperature_breaches"),

        round(
            avg("risk_score"),
            1
        ).alias("average_risk_score"),

        round(
            avg("late_arrival_hours"),
            2
        ).alias("average_late_arrival_hours"),

        spark_sum(
            coalesce(col("shipment_value_inr"), lit(0))
        ).alias("total_shipment_value_inr")
    )
    .withColumn(
        "high_risk_rate_pct",
        round(
            col("high_risk_shipment_count")
            / col("shipment_count") * 100,
            1
        )
    )
    .withColumn(
        "carrier_performance_category",
        when(col("high_risk_rate_pct") >= 40, "At Risk")
        .when(col("high_risk_rate_pct") >= 20, "Needs Attention")
        .otherwise("Stable")
    )
    .withColumn(
        "performance_calculated_at_utc",
        current_timestamp()
    )
)

overwrite_table(
    gold_carrier_performance,
    "gold_carrier_performance"
)

StatementMeta(, 7635f63e-1822-4c09-beea-52bc59275e2b, 7, Finished, Available, Finished, False)

Created gold_carrier_performance: 4 rows


In [6]:
from pyspark.sql.functions import (
    col,
    lit,
    when,
    coalesce,
    countDistinct,
    sum as spark_sum,
    avg,
    round,
    current_timestamp
)

# ============================================================
# GOLD: WAREHOUSE PERFORMANCE
# ============================================================

gold_warehouse_performance = (
    spark.table("gold_shipment_risk")
    .groupBy(
        "origin_warehouse_id"
    )
    .agg(
        countDistinct("shipment_id").alias("shipment_count"),

        spark_sum(
            when(
                col("shipment_status") == "Delivered",
                1
            ).otherwise(0)
        ).alias("delivered_shipment_count"),

        spark_sum(
            when(
                col("risk_level").isin("High", "Critical"),
                1
            ).otherwise(0)
        ).alias("high_risk_shipment_count"),

        spark_sum(
            coalesce(col("temperature_breach_count"), lit(0))
        ).alias("total_temperature_breaches"),

        round(
            avg("risk_score"),
            1
        ).alias("average_risk_score"),

        round(
            avg("late_arrival_hours"),
            2
        ).alias("average_late_arrival_hours"),

        spark_sum(
            coalesce(col("shipment_value_inr"), lit(0))
        ).alias("total_shipment_value_inr")
    )
    .withColumn(
        "high_risk_rate_pct",
        round(
            col("high_risk_shipment_count")
            / col("shipment_count") * 100,
            1
        )
    )
    .withColumn(
        "warehouse_risk_category",
        when(
            col("high_risk_rate_pct") >= 40,
            "At Risk"
        )
        .when(
            col("high_risk_rate_pct") >= 20,
            "Needs Attention"
        )
        .otherwise("Stable")
    )
    .withColumn(
        "performance_calculated_at_utc",
        current_timestamp()
    )
)

(
    gold_warehouse_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_warehouse_performance")
)

print(
    f"Created gold_warehouse_performance: "
    f"{gold_warehouse_performance.count()} rows"
)

StatementMeta(, 7635f63e-1822-4c09-beea-52bc59275e2b, 8, Finished, Available, Finished, False)

Created gold_warehouse_performance: 6 rows


In [7]:
display(
    spark.sql("""
        SELECT
            origin_warehouse_id,
            shipment_count,
            delivered_shipment_count,
            high_risk_shipment_count,
            high_risk_rate_pct,
            total_temperature_breaches,
            average_risk_score,
            average_late_arrival_hours,
            total_shipment_value_inr,
            warehouse_risk_category
        FROM gold_warehouse_performance
        ORDER BY average_risk_score DESC
    """)
)

StatementMeta(, 7635f63e-1822-4c09-beea-52bc59275e2b, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3290f0b7-6f94-451b-807c-c8d20eed70eb)

In [8]:
from pyspark.sql.functions import (
    col,
    lit,
    when,
    current_timestamp,
    round
)

# ============================================================
# GOLD: TEMPERATURE INCIDENTS
# ============================================================

gold_temperature_incidents = (
    spark.table("silver_sensor_readings")
    .join(
        shipment_limits,
        "shipment_id",
        "left"
    )
    .filter(
        (col("temperature_c") < col("allowed_min_temp_c")) |
        (col("temperature_c") > col("allowed_max_temp_c"))
    )
    .withColumn(
        "temperature_deviation_c",
        when(
            col("temperature_c") < col("allowed_min_temp_c"),
            col("allowed_min_temp_c") - col("temperature_c")
        )
        .otherwise(
            col("temperature_c") - col("allowed_max_temp_c")
        )
    )
    .withColumn(
        "incident_severity",
        when(
            col("temperature_deviation_c") >= 5,
            "Critical"
        )
        .when(
            col("temperature_deviation_c") >= 2,
            "High"
        )
        .otherwise(
            "Medium"
        )
    )
    .withColumn(
        "incident_created_at_utc",
        current_timestamp()
    )
    .select(
        "reading_id",
        "shipment_id",
        "sensor_device_id",
        "captured_at_utc",
        "temperature_c",
        "allowed_min_temp_c",
        "allowed_max_temp_c",
        round(
            col("temperature_deviation_c"),
            2
        ).alias("temperature_deviation_c"),
        "incident_severity",
        "latitude",
        "longitude",
        "incident_created_at_utc"
    )
)

(
    gold_temperature_incidents.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_temperature_incidents")
)

print(
    f"Created gold_temperature_incidents: "
    f"{gold_temperature_incidents.count()} rows"
)

StatementMeta(, 7635f63e-1822-4c09-beea-52bc59275e2b, 10, Finished, Available, Finished, False)

Created gold_temperature_incidents: 79 rows


In [9]:
display(
    spark.sql("""
        SELECT
            incident_severity,
            COUNT(*) AS incident_count
        FROM gold_temperature_incidents
        GROUP BY incident_severity
        ORDER BY
            CASE incident_severity
                WHEN 'Critical' THEN 1
                WHEN 'High' THEN 2
                ELSE 3
            END
    """)
)

StatementMeta(, 7635f63e-1822-4c09-beea-52bc59275e2b, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45b9127a-9a80-4cd5-ab3f-68380f54b2c9)

In [10]:
display(
    spark.sql("""
        SELECT
            reading_id,
            shipment_id,
            sensor_device_id,
            captured_at_utc,
            temperature_c,
            allowed_min_temp_c,
            allowed_max_temp_c,
            temperature_deviation_c,
            incident_severity
        FROM gold_temperature_incidents
        ORDER BY temperature_deviation_c DESC
    """)
)

StatementMeta(, 7635f63e-1822-4c09-beea-52bc59275e2b, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a54f58f-babc-442b-9bca-3aa340940b3d)